# USGS Water Data client (`georest.usgs.waterdata`) - usage examples

Walks through every public function in `georest.usgs.waterdata` against the live USGS Water
Data OGC API (`https://api.waterdata.usgs.gov/ogcapi/v1`), the modern replacement for the
NWIS Water Services:

- `list_collections`, `get_queryables` - what the API offers
- `search_monitoring_locations`, `get_monitoring_location` - find sites
- `get_time_series_metadata` - what a site measures
- `get_daily_values`, `get_continuous_values`, `get_latest_values` - observations
- `get_field_measurements`, `get_peaks` - site visits and annual peaks
- `get_items` and `to_rows` - any collection, any filter, as raw GeoJSON
- API keys, `max_items` and truncation, rate limits, and error handling

Every helper returns a **pandas DataFrame**: one row per observation for the time-series
collections, one per site for `monitoring-locations`, with `value` as a number and a site's
point as `lon` and `lat`. `get_items` is the raw escape hatch and returns the GeoJSON as sent.

**Rate limits.** Anonymous callers get "a few queries per hour". This notebook makes about 25
requests, so set `USGS_API_KEY` first (free key: https://api.waterdata.usgs.gov/signup/) or
run the cells selectively.

## Setup

```
pip install georest
```

or, from a clone of this repository, `pip install -e .` from the repo root. The chart and map
use `plotly` and `folium` (`pip install "georest[examples]"`).

In [1]:
import datetime as dt

import pandas as pd

import georest
from georest.usgs import waterdata as wd

pd.set_option("display.max_colwidth", None)
print("georest", georest.__version__)
print("base URL", wd.BASE_URL)

# Or set a key for this process only (never commit one):
# wd.set_api_key("your-key-here")

georest 0.5.0.dev0
base URL https://api.waterdata.usgs.gov/ogcapi/v1


Lees Ferry on the Colorado River (`USGS-09380000`), gauged continuously since 1921, is the
site used throughout. Bare ids (`"09380000"`) get the `USGS-` prefix automatically.

In [2]:
SITE = "USGS-09380000"
LEES_FERRY_BBOX = (-111.7, 36.8, -111.5, 37.0)   # west, south, east, north (lon/lat)

## What the API offers: `list_collections` and `get_queryables`

`COLLECTIONS` is the module's reference list of data collections; `list_collections()` is the
live, authoritative catalog (it also includes small code tables). `get_queryables` lists every
property that works as a direct filter on a collection.

In [3]:
collections = pd.DataFrame(wd.list_collections())
collections[["id", "title"]]

,id,title
0,latest-continuous,Latest continuous
1,continuous,Continuous values
2,latest-daily,Latest daily values
3,daily,Daily values
4,edr/daily,Daily values environmental data retrieval (EDR)
5,latest-field-measurements,Latest field measurements
6,field-measurements,Field measurements
7,channel-measurements,Channel measurements
8,combined-metadata,Combined metadata
9,field-measurements-metadata,Field measurements metadata


In [4]:
queryables = wd.get_queryables("daily")
pd.DataFrame(
    [(name, schema.get("type", ""), schema.get("title", "")) for name, schema in queryables.items()],
    columns=["property", "type", "title"],
)

,property,type,title
0,geometry,,
1,id,string,Record identifier
2,time_series_id,string,Time series ID
3,monitoring_location_id,string,Monitoring location ID
4,parameter_code,string,Parameter code
5,statistic_id,string,Statistic ID
6,time,string,Time
7,value,string,Value
8,unit_of_measure,string,Unit of measure
9,approval_status,string,Approval Status


## Find sites: `search_monitoring_locations`

Filters combine with AND. `site_type_code`: `ST` stream, `LK` lake, `GW` well, `SP` spring and
so on; a list of codes becomes a CQL2 `IN`. `hydrologic_unit_code` prefix-matches, so a HUC4
finds every site in the HUC8s and HUC12s beneath it.

In [5]:
# Streams inside a bounding box
sites = wd.search_monitoring_locations(bbox=LEES_FERRY_BBOX, site_type_code="ST")
sites[["id", "monitoring_location_name", "drainage_area", "lon", "lat"]]

,id,monitoring_location_name,drainage_area,lon,lat
0,USGS-09379950,"R200-01 11 MI ABV LEES FERRY IN GCNRA, AZ",NaN,-111.524603,36.897210
1,USGS-09379965,"R200-02 6.5 MI ABV LEES FERRY IN GCNRA, AZ.",NaN,-111.552659,36.873877
2,USGS-09380000,"COLORADO RIVER AT LEES FERRY, AZ",111800.0,-111.587872,36.864333
3,USGS-09380002,COLORADO R AT LEES FERRY - UNIT VALUES,111800.0,-111.588494,36.864988
4,USGS-09381950,"PARIA R. BLW. WATER POCKETS GULCH NR PAGE,AZ",NaN,-111.676276,36.946375
5,USGS-09382000,"PARIA RIVER AT LEES FERRY, AZ",1410.0,-111.594629,36.872211
6,USGS-09383000,"COLORADO RIVER AT COMPACT POINT NR LEES FERRY, AZ",112000.0,-111.606550,36.851376
7,USGS-09383003,"R200-03 AT RM5 BELOW 5 MILE WASH IN GCNP, AZ",NaN,-111.636550,36.807765
8,USGS-365041111323801,Tailwater -03,NaN,-111.544694,36.844894
9,USGS-365058111322200,WATER HOLES CANYON,NaN,-111.539333,36.849361


In [6]:
# Every stream and lake site in a HUC4 (prefix match), up to 200
huc_sites = wd.search_monitoring_locations(hydrologic_unit_code="1407",
                                           site_type_code=["ST", "LK"], max_items=200)
print(len(huc_sites), "sites; truncated:", huc_sites.attrs["truncated"])
huc_sites.groupby("site_type").size().rename("sites").to_frame()

200 sites; truncated: True


,sites
site_type,
"Lake, Reservoir, Impoundment",74
Stream,126


In [7]:
# By state and county FIPS (Utah, Kane County), trimming the response to a few properties
kane = wd.search_monitoring_locations(state_code="49", county_code="025",
                                      properties=["monitoring_location_name", "site_type"],
                                      max_items=20)
kane

,monitoring_location_name,site_type,id,lon,lat
0,(D-43-15) 2bad-1,Well,USEPA-370427111405201,-111.681111,37.074167
1,(D-43-10) 2bdc-1,Well,USEPA-370511111410001,-111.683333,37.086389
2,(C-43- 2) 5bcc-1,Well,USEPA-370609112232701,-112.390833,37.102500
3,(D-43- 2) 1bda-1,Well,USEPA-370615111460701,-111.769337,37.104151
4,(C-43- 4) 1baa-1,Well,USEPA-370628111545401,-111.915000,37.107778
5,(C-42-31) 6acd-1,Well,USEPA-370658112334201,-112.562428,37.116096
6,(C-42-30) 6dcc-1,Well,USEPA-370720112334601,-112.563539,37.122207
7,(C-42-30) 6dbc-1,Well,USEPA-370749112334801,-112.564095,37.124985
8,(C-42-30) 6baa-1,Well,USEPA-370809112340001,-112.567428,37.135818
9,(C-42-19) 6bca-1,Well,USEPA-370846112341701,-112.571389,37.146111


## One site: `get_monitoring_location`

Returns one flat record (a dict): every site attribute, plus `lon` and `lat`.

In [8]:
site = wd.get_monitoring_location("09380000")
pd.Series({k: site.get(k) for k in (
    "id", "monitoring_location_name", "site_type", "state_name", "county_name",
    "drainage_area", "altitude", "vertical_datum", "hydrologic_unit_code", "lon", "lat")})

id                                             USGS-09380000
monitoring_location_name    COLORADO RIVER AT LEES FERRY, AZ
site_type                                             Stream
state_name                                           Arizona
county_name                                  Coconino County
drainage_area                                       111800.0
altitude                                             3109.13
vertical_datum                                        NAVD88
hydrologic_unit_code                            140700061105
lon                                              -111.587872
lat                                                36.864333
dtype: object

## What a site measures: `get_time_series_metadata`

One row per (parameter, statistic, computation) with its `begin` and `end` dates and its
`time_series_id`. Check this before pulling values, so you ask for a parameter the site
actually reports.

In [9]:
series = wd.get_time_series_metadata(monitoring_location_id=SITE)
series[["parameter_code", "parameter_name", "statistic_id",
        "computation_period_identifier", "begin", "end"]].sort_values("parameter_code")

,parameter_code,parameter_name,statistic_id,computation_period_identifier,begin,end
0,00010,"Temperature, water",00002,Daily,1986-10-17T06:00:00+00:00,2026-10-26T06:00:00+00:00
21,00010,"Temperature, water",00001,Daily,1986-10-17T06:00:00+00:00,2026-10-26T06:00:00+00:00
4,00010,"Temperature, water",00003,Daily,1986-10-17T06:00:00+00:00,2026-10-27T06:00:00+00:00
16,00010,"Temperature, water",00011,Points,2007-10-01T07:00:00+00:00,2026-10-28T10:00:00+00:00
24,00045,Precipitation,NaN,Points,2026-06-29T00:00:00+00:00,2026-10-28T09:00:00+00:00
18,00060,Discharge,NaN,Water Year,1884-01-01T07:00:00+00:00,2023-04-25T06:00:00+00:00
15,00060,Discharge,00011,Points,1985-10-01T07:09:00+00:00,2026-10-28T10:00:00+00:00
13,00060,Discharge,00003,Daily,1921-10-01T07:00:00+00:00,2026-10-26T06:00:00+00:00
23,00065,Gage height,NaN,Water Year,1884-01-01T07:00:00+00:00,2023-04-25T06:00:00+00:00
19,00065,Gage height,00011,Points,2007-10-01T07:00:00+00:00,2026-10-28T10:00:00+00:00


## Daily statistics: `get_daily_values` (the old `/nwis/dv`)

Defaults: `parameter_code="00060"` (discharge, ft³/s) and `statistic_id="00003"` (daily mean).
`start` and `end` take strings, `date` or `datetime` objects, or an open side. Rows are sorted
by `(time_series_id, time)` **client-side**, because the API does not sort. Geometry is skipped
by default: it would be the same point on every row.

In [10]:
flow = wd.get_daily_values(SITE, "00060", start="2024-01-01", end="2024-01-31")
flow[["time", "value", "unit_of_measure", "approval_status", "qualifier"]]

,time,value,unit_of_measure,approval_status,qualifier
0,2024-01-01,11300.0,ft^3/s,Approved,None
1,2024-01-02,11900.0,ft^3/s,Approved,None
2,2024-01-03,11900.0,ft^3/s,Approved,None
3,2024-01-04,12000.0,ft^3/s,Approved,None
4,2024-01-05,12000.0,ft^3/s,Approved,None
5,2024-01-06,11900.0,ft^3/s,Approved,None
6,2024-01-07,11900.0,ft^3/s,Approved,None
7,2024-01-08,11900.0,ft^3/s,Approved,None
8,2024-01-09,11900.0,ft^3/s,Approved,None
9,2024-01-10,11900.0,ft^3/s,Approved,None


In [11]:
import plotly.express as px

px.line(flow, x="time", y="value", markers=True,
        labels={"time": "", "value": "Daily mean discharge (ft³/s)"},
        title="Colorado River at Lees Ferry, January 2024")

In [12]:
# Date objects, an open end (everything since June 1), and two statistics at once:
# daily maximum and minimum water temperature. (The series table above shows which daily
# series a site has; Lees Ferry reports gage height only as 15-minute readings.)
temps = wd.get_daily_values(SITE, "00010", statistic_id=["00001", "00002"],
                            start=dt.date(2024, 6, 1), end=None, max_items=100)
temps.pivot_table(index="time", columns="statistic_id", values="value").rename(
    columns={"00001": "max °C", "00002": "min °C"}).head(10)

statistic_id,max °C,min °C
time,,
2024-07-02,14.3,NaN
2024-07-03,NaN,12.5
2024-07-13,14.1,NaN
2024-07-17,NaN,11.8
2024-07-22,14.6,NaN
2024-07-28,NaN,12.0
2024-07-30,NaN,12.0
2024-07-31,14.1,NaN
2024-08-09,15.2,NaN


In [13]:
# Several sites at once: one call, a CQL2 IN under the hood
two = wd.get_daily_values(["09380000", "09402500"], start="2024-03-01", end="2024-03-07")
two.pivot_table(index="time", columns="monitoring_location_id", values="value")

monitoring_location_id,USGS-09380000,USGS-09402500
time,,
2024-03-01,11500.0,12400.0
2024-03-02,11000.0,11900.0
2024-03-03,10200.0,11200.0
2024-03-04,11100.0,10600.0
2024-03-05,11400.0,11700.0
2024-03-06,11400.0,11800.0
2024-03-07,11400.0,11900.0


## Sensor readings: `get_continuous_values` (the old `/nwis/iv`)

Typically every 15 minutes, so **about 35,000 rows per year per parameter**. The default
`max_items=10_000` covers roughly 100 days, and `df.attrs["truncated"]` is `True` when it bit.
`start="P2D"` is an ISO 8601 duration: the last two days. `time` carries a UTC offset.

In [14]:
stage = wd.get_continuous_values(SITE, "00065", start="P2D")
print(len(stage), "readings; truncated:", stage.attrs["truncated"])
stage[["time", "value", "unit_of_measure"]]

154 readings; truncated: False


,time,value,unit_of_measure
0,2026-10-26T22:45:00+00:00,8.06,ft
1,2026-10-26T23:00:00+00:00,8.06,ft
2,2026-10-26T23:15:00+00:00,8.08,ft
3,2026-10-26T23:30:00+00:00,8.07,ft
4,2026-10-26T23:45:00+00:00,8.08,ft
...,...,...,...
149,2026-10-28T21:00:00+00:00,8.02,ft
150,2026-10-28T21:15:00+00:00,8.02,ft
151,2026-10-28T21:30:00+00:00,8.03,ft
152,2026-10-28T21:45:00+00:00,8.03,ft


In [15]:
# A deliberately small cap, to see the truncated flag
capped = wd.get_continuous_values(SITE, "00060", start="P7D", max_items=50)
print(len(capped), "rows; truncated:", capped.attrs["truncated"])

50 rows; truncated: True


## The newest reading: `get_latest_values`

`source="daily"` (the default) or `"continuous"`. No site is required: a `bbox` and a
`parameter_code` give current conditions at every gage in an area, so geometry is kept here,
for a map.

In [16]:
wd.get_latest_values(SITE)[["parameter_code", "statistic_id", "time", "value", "unit_of_measure"]]

,parameter_code,statistic_id,time,value,unit_of_measure
0,00010,00002,2026-10-27,15.9,degC
1,63680,00002,2026-10-27,1.5,_FNU
2,00400,00008,2026-10-27,7.9,pH Units
3,00400,00001,2026-10-27,7.9,pH Units
4,00010,00003,2026-10-27,16.2,degC
5,63680,00001,2026-10-27,2.0,_FNU
6,80155,00003,1965-09-30,300.0,tons/day
7,80154,00003,1965-08-13,5.0,mg/l
8,00095,00001,2026-10-27,907.0,uS/cm
9,00060,00003,2026-10-26,6480.0,ft^3/s


In [17]:
# Current discharge at every gage in a box (continuous: the newest 15-minute reading)
now = wd.get_latest_values(source="continuous", parameter_code="00060",
                           bbox=(-112.5, 36.5, -111.0, 37.5))
now[["monitoring_location_id", "time", "value", "unit_of_measure", "lon", "lat"]]

,monitoring_location_id,time,value,unit_of_measure,lon,lat
0,USGS-09403690,1997-10-01T05:53:06+00:00,0.30,ft^3/s,-112.390755,37.127206
1,USGS-09382000,2026-10-28T22:00:00+00:00,23.90,ft^3/s,-111.594629,36.872211
2,USGS-09380000,2026-10-28T22:00:00+00:00,7800.00,ft^3/s,-111.587872,36.864333
3,USGS-09381500,2006-10-01T05:45:00+00:00,0.44,ft^3/s,-112.021576,37.481096
4,USGS-09381800,2026-10-28T22:10:00+00:00,15.10,ft^3/s,-111.906010,37.107484
5,USGS-09379910,2004-08-03T06:55:00+00:00,13500.00,ft^3/s,-111.483491,36.921655


In [18]:
import folium

m = folium.Map(location=[now["lat"].mean(), now["lon"].mean()], zoom_start=8,
               tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
               attr="Esri World Topo")
for _, row in now.iterrows():
    folium.CircleMarker(
        [row["lat"], row["lon"]], radius=6, color="#0b61a4", fill=True, fill_opacity=0.8,
        tooltip=f"{row['monitoring_location_id']}: {row['value']:,.0f} ft³/s at {row['time']}",
    ).add_to(m)
m

## Site visits and annual peaks: `get_field_measurements` and `get_peaks`

Field measurements are the manual readings taken on site visits (groundwater levels, and the
gage height and discharge measurements used to calibrate the sensors). Peaks are the annual
maximum streamflow, one per water year, with a `qualifier` list such as `["REGULATED"]`.

In [19]:
visits = wd.get_field_measurements(SITE, start="2023-01-01", max_items=20)
visits[["time", "parameter_code", "value", "unit_of_measure", "observing_procedure"]]

,time,parameter_code,value,unit_of_measure,observing_procedure
0,2023-12-06,00065,8.98,ft,None
1,2024-08-06,00060,15500.00,ft^3/s,Acoustic Doppler Current Profiler
2,2024-12-03,00065,8.56,ft,None
3,2025-02-04,00060,11200.00,ft^3/s,Acoustic Doppler Current Profiler
4,2025-02-04,00065,8.95,ft,None
5,2025-04-15,00065,8.70,ft,None
6,2025-04-15,00060,9320.00,ft^3/s,Acoustic Doppler Current Profiler
7,2025-04-15,00065,8.42,ft,"Gage height, staff gage"
8,2025-06-17,00065,8.92,ft,None
9,2025-08-12,00065,8.96,ft,"Gage height, staff gage"


In [20]:
peaks = wd.get_peaks(SITE, start="2010-01-01")
peaks[["water_year", "time", "value", "qualifier"]]

,water_year,time,value,qualifier
0,2010,2010-01-01,19200.00,[REGULATED]
1,2011,2011-07-07,26200.00,[REGULATED]
2,2012,2011-11-20,21200.00,[REGULATED]
3,2013,2012-11-20,44800.00,[REGULATED]
4,2014,2013-11-15,37700.00,[REGULATED]
5,2015,2014-11-11,38400.00,[REGULATED]
6,2016,2016-07-07,20400.00,[REGULATED]
7,2017,2016-11-09,36800.00,[REGULATED]
8,2018,2018-07-07,19000.00,[REGULATED]
9,2019,2018-11-06,39700.00,[REGULATED]


## Any collection, any filter: `get_items` and `to_rows`

The generic fetch under every helper, returning the GeoJSON FeatureCollection as the API sent
it. Use it for a collection without a helper, or with a CQL2 `filter`; any queryable property
works as a keyword equality filter. Unlike the helpers, it does not add `time` and `value`
back when you trim with `properties=`. `to_rows` flattens its output, and `georest.to_frame`
makes a DataFrame of it.

In [21]:
# A code table
georest.to_frame(wd.get_items("agency-codes", max_items=10))

,id,agency_name
0,AK001,Alaska Department of Transportation and Public Facilities
1,AK002,Alaska Department of Environmental Conservation
2,AK004,Alaska Department of Natural Resources (DNR)
3,AK008,Alaska Department of Fish and Game
4,AK010,"Alaska DNR, Division of Land and Water Management"
5,AK011,"Alaska DNR, Division of Geological and Geophysical Surveys"
6,AL001,Alabama Geological Survey
7,AL002,Alabama Water Improvement Commission
8,AL003,"Auburn University Water Resources Research Institute, AL"
9,AL006,Alabama State Highway Department


In [22]:
# A CQL2 text filter plus direct property filters on the daily collection
raw = wd.get_items("daily", monitoring_location_id=SITE, parameter_code="00060",
                   filter="statistic_id IN ('00001','00003') AND approval_status='Approved'",
                   start="2024-01-01", end="2024-01-05", skip_geometry=True,
                   properties="time,statistic_id,value")
pd.DataFrame(wd.to_rows(raw))

,time,statistic_id,value,id
0,2024-01-01,00003,11300.0,c0beb6eb-42fe-4889-9c36-61ad8080f007
1,2024-01-02,00003,11900.0,8e631e1b-88be-47c3-b491-a874d924c3d2
2,2024-01-03,00003,11900.0,1b63d564-f110-4ad5-bddf-21fbe44dd373
3,2024-01-04,00003,12000.0,46472b83-e34a-41c1-9c59-82a65cccc72a
4,2024-01-05,00003,12000.0,b2a6aaf1-9d02-4ae0-b53d-2bdc8676079e


## API keys, rate limits and errors

Resolution per call: an explicit `api_key=` (`""` forces anonymous), then `set_api_key(...)`,
then `USGS_API_KEY` or `USGS_WATERDATA_API_KEY` in the environment, then anonymous. The key is
sent **only** as the `X-Api-Key` header, never in a URL, so it cannot appear in an error
message. When the API reports the remaining hourly budget, `df.attrs["rate_limit"]` carries it.

In [23]:
df = wd.get_daily_values(SITE, start="2024-01-01", end="2024-01-02")
print("rate_limit:", df.attrs.get("rate_limit"))   # absent when the server sent no X-RateLimit headers

rate_limit: None


In [24]:
# A bad request surfaces the API's own description in the RuntimeError message
try:
    wd.get_daily_values(SITE, start="not-a-date")
except RuntimeError as exc:
    print(type(exc).__name__)
    print(exc)

RuntimeError
Request failed (400): https://api.waterdata.usgs.gov/ogcapi/v1/collections/daily/items?datetime=not-a-date/..&skipGeometry=true&monitoring_location_id=USGS-09380000&parameter_code=00060&statistic_id=00003&limit=10000&f=json
{
  "code": "InvalidParameterValue",
  "type": "InvalidParameterValue",
  "description": "Unknown string format: not-a-date"
}


In [25]:
# Reserved or invalid arguments are refused before any request is made
for bad in (lambda: wd.get_daily_values(SITE, cursor="x"),
            lambda: wd.get_daily_values(SITE, page_size=0),
            lambda: wd.get_latest_values(source="hourly"),
            lambda: wd.get_daily_values([])):
    try:
        bad()
    except (TypeError, ValueError) as exc:
        print(f"{type(exc).__name__}: {exc}")

TypeError: cursor is set by the client; use limit=/max_items= instead of passing it as a filter
ValueError: limit must be an int in 1..50000, got 0
ValueError: source must be one of ['continuous', 'daily'], got 'hourly'
ValueError: monitoring_location_id list is empty


In [26]:
# A 429 the client will not wait out raises RateLimitError (a RuntimeError subclass) with
# .retry_after: catch it by name and back off rather than looping.
try:
    df = wd.get_daily_values(SITE, start="2020-01-01", end="2020-01-02")
    print("ok:", len(df), "rows")
except wd.RateLimitError as exc:
    print("rate limited; retry after", exc.retry_after, "seconds")

ok: 2 rows


## Why the helpers sort

The API returns rows in storage order (a January window can come back starting on the 22nd).
Every time-series helper sorts by `(time_series_id, time)`; `get_items` does not.

In [27]:
unsorted = wd.get_items("daily", monitoring_location_id=SITE, parameter_code="00060",
                        statistic_id="00003", start="2020-01-01", end="2020-01-31",
                        skip_geometry=True, properties="time,value")
helper = wd.get_daily_values(SITE, "00060", start="2020-01-01", end="2020-01-31")
# The raw features in the order the API sent them (to_rows would sort them)
pd.DataFrame({"get_items order": [f["properties"]["time"] for f in unsorted["features"]],
              "helper order": list(helper["time"])})

,get_items order,helper order
0,2020-01-22,2020-01-01
1,2020-01-27,2020-01-02
2,2020-01-24,2020-01-03
3,2020-01-04,2020-01-04
4,2020-01-14,2020-01-05
5,2020-01-03,2020-01-06
6,2020-01-05,2020-01-07
7,2020-01-13,2020-01-08
8,2020-01-30,2020-01-09
9,2020-01-21,2020-01-10
